# 01 — Knesset OData discovery

Before writing a pipeline, profile the source by hand. This notebook answers:

1. **Connection** — what does the API expose and how do we call it?
2. **Schema** — which tables, keys and columns are there?
3. **Volume** — how many rows per table, and how much is in our scope (20th Knesset onward)?
4. **Paging** — how does the API page, and how fast?
5. **Change tracking** — can we load only what changed?
6. **Data** — what do the rows look like: types, nulls, codes, quirks?

Each section ends with a **finding** that drove a decision in `ingestion/odata.py`.

Run it from the repo root with `uv run --group notebook jupyter lab`. Section 6 reads `data/raw_sample.duckdb`, so run `make sample` first.

In [1]:
import sys
import time
import xml.etree.ElementTree as ET
from pathlib import Path

import duckdb
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from dlt.sources.helpers import requests  # noqa: E402

from ingestion.odata import TABLES, Api, load_settings, scope  # noqa: E402

pd.set_option("display.max_columns", 30, "display.width", 200)

settings = load_settings()  # ingestion/.dlt/config.toml
api = Api(settings)  # retries and backoff, as in the pipeline
BASE_URL = settings.base_url


def get_json(url: str, params: dict[str, str]) -> dict:
    response = api.session.get(url, params=params)
    response.raise_for_status()
    return response.json()


BASE_URL

'https://knesset.gov.il/OdataV4/ParliamentInfo/'

## 1. Connection

The root URL returns the *service document*: the list of entity sets (tables). No authentication is needed.

In [2]:
service = get_json(BASE_URL, {})
entity_sets = [e["name"] for e in service["value"]]
print(len(entity_sets), "entity sets")
entity_sets

48 entity sets


['KNS_Agenda',
 'KNS_Bill',
 'KNS_BillHistoryInitiator',
 'KNS_BillInitiator',
 'KNS_BillName',
 'KNS_BillSplit',
 'KNS_BillUnion',
 'KNS_BroadcastCommitteSession',
 'KNS_CmtSessionItem',
 'KNS_CmtSiteCode',
 'KNS_Committee',
 'KNS_CommitteeSession',
 'KNS_DocumentAgenda',
 'KNS_DocumentBill',
 'KNS_DocumentCommitteeSession',
 'KNS_DocumentIsraelLaw',
 'KNS_DocumentPlenumSession',
 'KNS_DocumentQuerie',
 'KNS_DocumentSecondaryLaw',
 'KNS_Faction',
 'KNS_GovMinistry',
 'KNS_IsraelLaw',
 'KNS_IsraelLawBinding',
 'KNS_IsraelLawClassificiation',
 'KNS_IsraelLawLawCorrections',
 'KNS_IsraelLawMinistry',
 'KNS_IsraelLawName',
 'KNS_ItemType',
 'KNS_JointCommittee',
 'KNS_KnessetDates',
 'KNS_LawBinding',
 'KNS_LawCorrections',
 'KNS_MkSiteCode',
 'KNS_Person',
 'KNS_PersonToPosition',
 'KNS_PlenumSession',
 'KNS_PlenumVote',
 'KNS_PlenumVoteResult',
 'KNS_PlmSessionItem',
 'KNS_Position',
 'KNS_Query',
 'KNS_SecLawAuthorizingLaw',
 'KNS_SecLawRegulator',
 'KNS_SecondaryLaw',
 'KNS_SecToSecBi

## 2. Schema

`$metadata` is an XML document describing every entity type: its key and typed properties.

In [3]:
EDM = {"e": "http://docs.oasis-open.org/odata/ns/edm"}
metadata = ET.fromstring(requests.get(BASE_URL + "$metadata", timeout=120).content)

columns = pd.DataFrame(
    [
        {
            "entity": t.get("Name"),
            "column": p.get("Name"),
            "type": p.get("Type").removeprefix("Edm."),
            "nullable": p.get("Nullable") != "false",
        }
        for t in metadata.iter(f"{{{EDM['e']}}}EntityType")
        for p in t.iterfind("e:Property", EDM)
    ]
)
keys = {
    t.get("Name"): [k.get("Name") for k in t.iterfind("e:Key/e:PropertyRef", EDM)]
    for t in metadata.iter(f"{{{EDM['e']}}}EntityType")
}

schema = columns.groupby("entity").agg(
    n_columns=("column", "size"),
    has_last_updated=("column", lambda c: "LastUpdatedDate" in set(c)),
    has_knesset_num=("column", lambda c: "KnessetNum" in set(c)),
)
schema["key"] = schema.index.map(lambda e: ",".join(keys[e]))
schema

,n_columns,has_last_updated,has_knesset_num,key
entity,,,,
KNS_Agenda,20,True,True,Id
KNS_Bill,23,True,True,Id
KNS_BillHistoryInitiator,9,True,False,Id
KNS_BillInitiator,6,True,False,Id
KNS_BillName,6,True,False,Id
KNS_BillSplit,5,True,False,Id
KNS_BillUnion,4,True,False,Id
KNS_BroadcastCommitteSession,3,False,False,Id
KNS_CmtSessionItem,8,True,False,Id


In [4]:
print("Key is always Id:", (schema["key"] == "Id").all())
print("Tables without LastUpdatedDate:", list(schema.index[~schema.has_last_updated]))
columns[columns.entity == "KNS_Bill"]

Key is always Id: True
Tables without LastUpdatedDate: ['KNS_BroadcastCommitteSession', 'KNS_CmtSiteCode', 'KNS_ItemType', 'KNS_MkSiteCode', 'V_Lobbyist', 'V_LobbyistsClient']


,entity,column,type,nullable
20,KNS_Bill,Id,Int32,False
21,KNS_Bill,KnessetNum,Int32,True
22,KNS_Bill,Name,String,True
23,KNS_Bill,TypeID,Int32,True
24,KNS_Bill,TypeDesc,String,True
25,KNS_Bill,SubTypeID,Int32,True
26,KNS_Bill,SubTypeDesc,String,True
27,KNS_Bill,PrivateNumber,Int32,True
28,KNS_Bill,CommitteeID,Int32,True
29,KNS_Bill,StatusID,Int32,True


**Finding.** Every table has a single integer key `Id`, and all but 6 small lookup/view tables have `LastUpdatedDate`.
→ merge on `Id`, load incrementally on `LastUpdatedDate`, fully replace the rest (`KNS_ItemType`).

## 3. Volume

`$count=true&$top=0` returns the row count without any rows. We compare the whole table with our scope filter.

In [5]:
def count(table: str, where: str | None = None) -> int:
    params = {"$count": "true", "$top": "0"}
    if where:
        params["$filter"] = where
    return get_json(BASE_URL + table, params)["@odata.count"]


volume = pd.DataFrame(
    [
        {
            "table": name,
            "all_rows": count(name),
            "in_scope": count(name, scope(name, settings)),
            "scope": scope(name, settings) or "(all)",
        }
        for name, t in TABLES.items()
    ]
).set_index("table")
volume["share"] = (volume.in_scope / volume.all_rows).round(2)
volume.sort_values("in_scope", ascending=False)

,all_rows,in_scope,scope,share
table,,,,
KNS_PlenumVoteResult,1953709,1354705,VoteDate ge 2015-03-31T00:00:00+03:00,0.69
KNS_BillInitiator,169348,169348,(all),1.00
KNS_PlmSessionItem,168802,168802,(all),1.00
KNS_CmtSessionItem,80790,80790,(all),1.00
KNS_CommitteeSession,109276,26476,KnessetNum ge 20,0.24
KNS_Bill,60137,23115,KnessetNum ge 20,0.38
KNS_PlenumVote,36168,19686,VoteDateTime ge 2015-03-31T00:00:00+03:00,0.54
KNS_PersonToPosition,23971,8263,KnessetNum ge 20,0.34
KNS_Agenda,22072,5062,KnessetNum ge 20,0.23


In [6]:
volume[["all_rows", "in_scope"]].sum()

all_rows    2683816
in_scope    1867105
dtype: int64

**Finding.** Plenum vote results (one row per MK per vote) dominate: ~1.35M of ~1.9M in-scope rows (2.7M in total).
Limiting to the 20th Knesset (2015+) cuts the load by about a third and still covers the committee protocols (Nov 2018+).
Three link tables (`KNS_BillInitiator`, `KNS_CmtSessionItem`, `KNS_PlmSessionItem`) have no Knesset/date column, so they load in full and get filtered in dbt.

## 4. Paging

A plain request returns 100 rows and an `@odata.nextLink`. `$top` above 100 is capped.

In [7]:
page = get_json(BASE_URL + "KNS_Bill", {"$top": "1000"})
len(page["value"]), page.get("@odata.nextLink")

(100,
 'https://knesset.gov.il/OdataV4/ParliamentInfo/KNS_Bill?$top=900&$skip=100')

The next link pages with `$skip`. On a large table the server must skip all earlier rows, so deep pages get slow.
Keyset paging (`Id gt <last id>` ordered by `Id`) jumps straight to the right place.

In [8]:
def seconds(params: dict[str, str]) -> float:
    start = time.perf_counter()
    get_json(BASE_URL + "KNS_PlenumVoteResult", params)
    return round(time.perf_counter() - start, 2)


paging = pd.DataFrame(
    [
        {
            "rows_in": n,
            "skip_seconds": seconds({"$skip": str(n)}),
            "keyset_seconds": seconds({"$filter": f"Id gt {n}", "$orderby": "Id"}),
        }
        for n in [0, 500_000, 1_500_000]
    ]
)
paging

,rows_in,skip_seconds,keyset_seconds
0,0,1.80,0.46
1,500000,3.23,0.32
2,1500000,8.84,0.43


**Finding.** `$skip` gets slower with depth (3–20s per page deep in the vote table across our runs); keyset stays around a second.
→ the pipeline pages by `Id`, not by `@odata.nextLink`. At ~1s per 100 rows, a full in-scope load takes ~3 hours, mostly vote results.

## 5. Change tracking

How many rows changed recently? This is what a daily incremental run would fetch.

In [9]:
since = (pd.Timestamp.now(tz="Asia/Jerusalem") - pd.Timedelta(days=7)).floor("s").isoformat()
recent = pd.Series(
    {
        name: count(
            name, " and ".join(filter(None, [scope(name, settings), f"LastUpdatedDate ge {since}"]))
        )
        for name, t in TABLES.items()
        if t.incremental
    },
    name="changed_last_7_days",
)
recent.sort_values(ascending=False)

KNS_CmtSessionItem      13740
KNS_PlmSessionItem      13555
KNS_CommitteeSession    10855
KNS_Bill                 7493
KNS_Query                1580
KNS_PersonToPosition     1285
KNS_Agenda                798
KNS_PlenumSession         419
KNS_Person                119
KNS_Committee              85
KNS_Faction                21
KNS_Position                0
KNS_GovMinistry             0
KNS_KnessetDates            0
KNS_BillInitiator           0
KNS_PlenumVote              0
KNS_IsraelLaw               0
KNS_PlenumVoteResult        0
KNS_Status                  0
Name: changed_last_7_days, dtype: int64

In [10]:
sample = get_json(BASE_URL + "KNS_Person", {"$top": "5", "$orderby": "Id"})["value"]
[row["LastUpdatedDate"] for row in sample]

['2024-07-28T21:41:35.37+03:00',
 '2020-03-23T23:33:55.387+02:00',
 '2020-03-23T23:40:56.46+02:00',
 '2020-11-19T13:43:11.277+02:00',
 '2020-03-23T23:40:27.3+02:00']

**Finding.** About 50K rows changed in the last 7 days, against 1.9M in scope, and none in the vote tables, so incremental runs take minutes instead of hours.
Timestamps carry a local offset that switches between `+02:00` (winter) and `+03:00` (summer), so comparing them as strings is wrong.
→ `parse_row` turns `LastUpdatedDate` into a real datetime before dlt tracks it.

Some rows have old business dates but a recent `LastUpdatedDate` (bulk back-office updates), so "changed" doesn't mean "new".

## 6. Data profile (sample)

From here on we read the sample load (`make sample`: up to 200 rows per table in `data/raw_sample.duckdb`).
dlt renamed everything to snake_case: `KNS_Bill.KnessetNum` → `bronze.kns_bill.knesset_num`.
DuckDB's `SUMMARIZE` gives types, min/max, distinct counts and null share per column.

In [11]:
db = duckdb.connect(str(ROOT / "data" / "raw_sample.duckdb"), read_only=True)
db.sql("""
    select table_name, estimated_size as rows, column_count
    from duckdb_tables() where schema_name = 'bronze' and table_name not like '\\_dlt%' escape '\\'
    order by table_name
""").df()

,table_name,rows,column_count
0,deleted_ids,0,4
1,kns_agenda,200,24
2,kns_bill,200,27
3,kns_bill_initiator,200,10
4,kns_cmt_session_item,200,12
5,kns_committee,200,20
6,kns_committee_session,200,19
7,kns_faction,132,11
8,kns_gov_ministry,200,11
9,kns_israel_law,200,19


In [12]:
db.sql("summarize bronze.kns_bill").df()[
    ["column_name", "column_type", "min", "max", "approx_unique", "null_percentage"]
]

,column_name,column_type,min,max,approx_unique,null_percentage
0,id,BIGINT,482355,562357,211,0.0
1,knesset_num,BIGINT,20,20,1,0.0
2,name,VARCHAR,הצעת חוק אימוץ ילדים (תיקון - אימוץ על ידי בני...,חוק תגמול לנושאי משרה בתאגידים פיננסיים (אישור...,203,0.0
3,type_id,BIGINT,2,2,1,0.0
4,type_desc,VARCHAR,הצעת חוק,הצעת חוק,1,0.0
5,sub_type_id,BIGINT,53,54,2,0.0
6,sub_type_desc,VARCHAR,ממשלתית,פרטית,2,0.0
7,private_number,BIGINT,1,2391,197,20.0
8,committee_id,BIGINT,-1,935,11,0.0
9,status_id,BIGINT,104,177,7,0.0


In [13]:
db.sql("summarize bronze.kns_committee_session").df()[
    ["column_name", "column_type", "min", "max", "approx_unique", "null_percentage"]
]

,column_name,column_type,min,max,approx_unique,null_percentage
0,id,BIGINT,562190,566499,205,0.0
1,number,BIGINT,1,42,27,7.5
2,knesset_num,BIGINT,20,20,1,0.0
3,type_id,BIGINT,160,161,2,0.0
4,type_desc,VARCHAR,חסויה,פתוחה,2,0.0
5,committee_id,BIGINT,922,945,16,0.0
6,status_id,BIGINT,192,193,2,0.0
7,status_desc,VARCHAR,מבוטלת,פעילה,2,0.0
8,location,VARCHAR,אולם ירושלים,כלכלה,7,0.0
9,session_url,VARCHAR,http://main.knesset.gov.il/Activity/committees...,http://main.knesset.gov.il/Activity/committees...,194,0.0


### Codes and lookups

Many tables carry a code plus its Hebrew text (`type_id` / `type_desc`). Status codes live in `KNS_Status`, grouped by what they describe.

In [14]:
db.sql(
    "select type_desc, count(*) as statuses from bronze.kns_status group by 1 order by 2 desc"
).df()

,type_desc,statuses
0,הצעת חוק,36
1,הצעה לסדר היום,11
2,פעולה על פי חוק,10
3,פריט ועדה,6
4,שאילתה,4
5,ישיבת ועדה,4
6,ישיבת מליאה,3
7,פריטי מליאה,3
8,חוק בן חיצוני,2
9,חוק אב,1


In [15]:
db.sql('select id, "desc", table_name from bronze.kns_item_type order by id').df()

,id,desc,table_name
0,1,שאילתה,KNS_Query
1,2,הצעת חוק,KNS_Bill
2,4,הצעה לסדר היום,KNS_Agenda
3,5,ישיבת מליאה,KNS_PlenumSession
4,12,ישיבת ועדה,KNS_CommitteSession
5,6000,פעולה על פי חוק,KNS_SecondaryLaw
6,6002,חוק בן חיצוני,KNS_Bill
7,6003,חוק אב,KNS_IsraelLaw


In [16]:
db.sql(
    "select result_code, result_desc, count(*) as n from bronze.kns_plenum_vote_result group by all order by 1"
).df()

,result_code,result_desc,n
0,7,בעד,157
1,8,נגד,43


### Quirks to handle in dbt staging

In [17]:
# One row per person per position period: the same person appears many times (MK, minister, committee member...)
db.sql("""
    select position_id, count(*) as n, count(distinct person_id) as people
    from bronze.kns_person_to_position group by 1 order by 2 desc limit 8
""").df()

,position_id,n,people
0,43,91,91
1,42,42,36
2,61,29,29
3,66,14,14
4,39,13,10
5,70,2,2
6,40,2,2
7,57,2,2


In [18]:
# Vote results repeat the MK's name; the person table is the source of truth
# (person_in_sample is False only because the sample holds just 200 people)
db.sql("""
    select r.mk_id, r.first_name || ' ' || r.last_name as name_on_vote, p.id is not null as person_in_sample
    from bronze.kns_plenum_vote_result r left join bronze.kns_person p on p.id = r.mk_id
    limit 5
""").df()

,mk_id,name_on_vote,person_in_sample
0,30555,לאה פדידה,False
1,23632,מיקי לוי,False
2,23512,אילת שקד,False
3,23591,אלעזר שטרן,False
4,4397,זאב אלקין,False


## Findings → decisions

| Finding | Decision |
| --- | --- |
| Every table keyed by integer `Id`; most have `LastUpdatedDate` | merge on `Id`, incremental on `LastUpdatedDate`, replace lookup tables |
| `$skip` paging slows down with depth | keyset paging: `Id gt <last>` + `$orderby=Id` |
| Max 100 rows per request, ~1s each | full load takes hours → scope to the 20th Knesset onward, sample mode for testing |
| Vote results are ~70% of in-scope rows | date filter on vote tables; nothing else to trim (all rows are real votes) |
| Timestamps switch between `+02:00` and `+03:00` | parse to datetime before tracking increments |
| `$apply` (server-side group by) is blocked by the site's firewall | aggregate in DuckDB, not in the API |
| Codes come with Hebrew text; `KNS_Status` groups statuses by type | decode and translate in dbt staging with seeds |
| Vote rows repeat MK names; link tables have no Knesset column | take names from `KNS_Person`; filter link tables by their parent in dbt |